# 05 — Export SEC artefacts for external projects

This notebook is the **reference for how external projects should consume
SEC outputs** (e.g. the `caliper_video` project that overlays caliper logs,
breakouts, and now SEC breakpoints).

It uses only `karst_analysis.sec.export` — the stable public API. Internal
modules (`io`, `preprocessing`, `breakpoints`) should not be imported from
external projects: their internals may change, the export module won't.

## What you can do here

1. Discover what's available on disk: `list_available_runs()`.
2. Load a smoothed SEC profile (in both datums): `load_sec_profile()`.
3. Load breakpoints for any N: `load_breakpoints_at_n()`.
4. Load the BIC curve to inform the N decision: `load_bic_curve()`.

The point of `load_breakpoints_at_n` is that **you can change N freely**
without re-running anything — the BIC sweep with N=1..10 has already
been computed and stored in JSON. You decide the N when you cross-check
with caliper and video.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd
import matplotlib.pyplot as plt

from karst_analysis.sec.export import (
    list_available_runs,
    load_sec_profile,
    load_breakpoints_at_n,
    load_bic_curve,
)

# Make tables show all columns
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print(f"CWD: {Path.cwd()}")

## 1. What's available?

In [ ]:
inv = list_available_runs(campaign="2022_02")
inv

## 2. Load a smoothed SEC profile

Always returns both `depth_m` (water-table datum) and `depth_bgl_m`
(ground-level datum, same as caliper). Pick whichever you need.

In [ ]:
sec = load_sec_profile(
    well_id="LRS70D",
    campaign="2022_02",
    smoothing="lowess",
)
print(f"shape: {sec.shape}")
print(f"columns: {sec.columns.tolist()}")
sec.head()

## 3. Load breakpoints at any N

Try a few values — change `n` freely and re-run.

In [ ]:
# >>> Change this to whatever N you're considering <<<
N_TO_TRY = 3

bps = load_breakpoints_at_n(
    well_id="LRS70D",
    campaign="2022_02",
    smoothing="lowess",
    n=N_TO_TRY,
)
bps

**Read those CIs carefully**:

- `ci_width_m` < 0.2 m → very confident BP, the algorithm pinned the depth tightly.
- `ci_width_m` ~ 0.5 – 1 m → moderately confident.
- `ci_width_m` > 1 m → flaky BP, treat as suggestive only.

When deciding which N to keep at the end, **prefer Ns where most BPs have
narrow CIs**. A high-N fit with one or two BPs that have huge CIs is the
algorithm telling you it's overfitting.

## 4. The BIC curve (decision aid)

Lower BIC = better. Look for an "elbow" — the point past which adding
more breakpoints stops giving meaningful gain.

In [ ]:
curve_savgol = load_bic_curve(
    well_id="LRS70D", campaign="2022_02", smoothing="savgol")
curve_lowess = load_bic_curve(
    well_id="LRS70D", campaign="2022_02", smoothing="lowess")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curve_savgol["n_breakpoints"], curve_savgol["bic"],
        marker="o", color="#c0392b", label="savgol")
ax.plot(curve_lowess["n_breakpoints"], curve_lowess["bic"],
        marker="o", color="#1f4e79", label="lowess")
ax.set_xlabel("n_breakpoints")
ax.set_ylabel("BIC (lower is better)")
ax.set_title("LRS70D — BIC vs N")
ax.grid(True, ls=":", alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Example: overlay SEC + breakpoints on a caliper plot

This is a **mock**: shows how an external project would take the loaded
data and produce a multi-panel figure. Replace the synthetic caliper
data with the real caliper trace from `caliper_video`.

In [ ]:
import numpy as np

# === Load SEC ===
sec = load_sec_profile(
    well_id="LRS70D", campaign="2022_02", smoothing="lowess")
bps = load_breakpoints_at_n(
    well_id="LRS70D", campaign="2022_02", smoothing="lowess", n=5)

# === Mock caliper data (REPLACE with real load from caliper_video) ===
np.random.seed(0)
caliper_depth = np.linspace(0, 30, 600)
caliper_diam = 6 + 0.05*np.sin(caliper_depth*0.7) + 0.02*np.random.randn(600)
# Mock breakouts at a few depths to mimic the caliper_video output
breakout_depths = [4.6, 8.7, 12.6, 22.0]

# === Plot ===
fig, (ax_cal, ax_sec) = plt.subplots(
    1, 2, figsize=(11, 9), sharey=True, gridspec_kw={"width_ratios": [1, 2]})

# Caliper panel
ax_cal.plot(caliper_diam, caliper_depth, color="#2c3e50", lw=1)
for bd in breakout_depths:
    ax_cal.axhline(bd, color="#c0392b", ls="--", lw=0.8, alpha=0.6)
ax_cal.set_xlabel("Caliper diameter [in] (mock)")
ax_cal.set_ylabel("Depth [m, BGL]")
ax_cal.set_title("Caliper + breakouts")
ax_cal.grid(True, ls=":", alpha=0.5)

# SEC panel — note: using depth_bgl_m to match the caliper datum
ax_sec.plot(sec["sec_uS_cm"], sec["depth_bgl_m"],
            color="#1f4e79", lw=1.2, label="SEC (lowess)")

# Breakpoint markers + CI bars in BGL
for _, bp in bps.iterrows():
    # CI bar (vertical)
    ax_sec.vlines(bp["sec_at_bp_log10"], bp["ci_lower_bgl_m"], bp["ci_upper_bgl_m"],
                  color="#e67e22", lw=2, alpha=0.5)
    # Marker
    ax_sec.scatter(10**bp["sec_at_bp_log10"], bp["depth_bgl_m"],
                   s=120, marker="D",
                   facecolor="#e67e22", edgecolor="black", zorder=5,
                   label=f"BP{bp['bp_index']}: {bp['depth_bgl_m']:.2f} m"
                   if bp["bp_index"] == 1 else None)
    ax_sec.annotate(
        f"BP{int(bp['bp_index'])}\n{bp['depth_bgl_m']:.2f} m\n"
        f"CI ±{bp['ci_width_m']/2:.2f}",
        (10**bp["sec_at_bp_log10"], bp["depth_bgl_m"]),
        textcoords="offset points", xytext=(8, 0),
        fontsize=8, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.3",
                  facecolor="white", edgecolor="#e67e22", alpha=0.9),
    )

# Mock breakouts on the SEC panel for visual cross-check
for bd in breakout_depths:
    ax_sec.axhline(bd, color="#c0392b", ls="--", lw=0.6, alpha=0.4)

ax_sec.set_xlabel("SEC [µS/cm]")
ax_sec.set_title("SEC (lowess) + breakpoints (N=5)")
ax_sec.invert_yaxis()
ax_cal.invert_yaxis()  # ensure consistent inversion
ax_sec.grid(True, ls=":", alpha=0.5)
ax_sec.legend(loc="lower right", fontsize=8)

fig.suptitle("LRS70D — caliper + SEC convergence (mock)", fontsize=13, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## 6. Quick decision loop: try several N values

Loop over a few candidate N's and produce one figure each. **In your
real workflow with caliper_video, you'd compare each candidate against
the caliper breakouts and the video annotations to pick the N that
maximises convergence.**

In [ ]:
for n_try in [2, 3, 5, 8]:
    bps_n = load_breakpoints_at_n(
        well_id="LRS70D", campaign="2022_02", smoothing="lowess", n=n_try)
    print(f"\n=== N = {n_try} (BIC = {bps_n['bic'].iloc[0]:.1f}) ===")
    # Just print the depths in BGL for quick inspection
    summary = bps_n[["bp_index", "depth_bgl_m", "ci_lower_bgl_m",
                     "ci_upper_bgl_m", "ci_width_m"]].copy()
    print(summary.to_string(index=False))

## 7. From an EXTERNAL project (e.g. caliper_video)

In that other project's code, you'd add the karst_analysis package as
a dependency (editable install or PyPI), and then:

```python
from karst_analysis.sec.export import (
    load_sec_profile, load_breakpoints_at_n,
)

# `project_root` points to where data/, results/, etc. live on disk.
# If both projects share the same parent folder, pass that path here.
KARST_ROOT = "/path/to/karst_analysis_repo"

sec  = load_sec_profile(
    well_id="LRS70D", campaign="2022_02",
    smoothing="lowess", project_root=KARST_ROOT)
bps  = load_breakpoints_at_n(
    well_id="LRS70D", campaign="2022_02",
    smoothing="lowess", n=5, project_root=KARST_ROOT)

# Then plot bps['depth_bgl_m'] alongside your caliper data — same datum.
```

That's the contract. The `project_root` argument is the only thing that
needs to know about cross-project paths.
